In [ ]:
%load_ext cudf.pandas

In [ ]:
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd

In [ ]:
%%time
### cell 0 ###

course = pd.read_csv(
    "/home/dias-benchmarks/notebooks/aieducation/what-course-are-you-going-to-take/input/course-reviews-university-of-waterloo/course_data_clean.csv"
)
factor = 300
course = pd.concat([course] * factor, ignore_index=True)
course.info()

In [ ]:
%%time
### cell 1 ###

course.head(10)

In [ ]:
%%time
### cell 2 ###

course.tail(10)

In [ ]:
%%time
### cell 3 ###

course.describe()

In [ ]:
%%time
### cell 4 ###

course.info()

In [ ]:
%%time
### cell 5 ###

course = course.dropna()

In [ ]:
%%time
### cell 6 ###

course[["course_unit", "course_num"]] = course["course_code"].str.split(
    " ", expand=True
)

In [ ]:
%%time
### cell 7 ###

course

In [ ]:
%%time
### cell 8 ###

course[course["num_reviews"] < 10].index

In [ ]:
%%time
### cell 9 ###


def extract_filter_features(df, col_name, threshold):
    """Extracts features for predicting execution time of filtering operation."""
    s = df[col_name]
    num_rows = len(s)
    dtype_str = str(s.dtype)
    # avoid dropping/branching; use reductions directly
    # use a safe denominator to avoid division by zero
    denom = num_rows if num_rows else 1
    nan_ratio = s.isna().sum() / denom
    target_selectivity = s.lt(threshold).sum() / denom

    return {
        "num_rows": num_rows,
        "dtype_str": dtype_str,
        "nan_ratio": nan_ratio,
        "target_selectivity": target_selectivity,
    }


# call remains the same; all ops execute on GPU
extract_filter_features(course, "num_reviews", 10)

In [ ]:
%%time
### cell 10 ###

course.drop(course[course["num_reviews"] < 10].index, inplace=True)

In [ ]:
%%time
### cell 11 ###

course

In [ ]:
%%time
### cell 12 ###

for i in ["useful", "easy", "liked"]:
    course[i] = course[i].str.replace("%", "")
    course[i] = course[i].astype("int")

In [ ]:
%%time
### cell 13 ###

course.set_index("course_unit", inplace=True)

In [ ]:
%%time
### cell 14 ###

course

In [ ]:
%%time
### cell 15 ###

course.drop(["course_title", "reviews", "course_rating"], axis=1, inplace=True)

In [ ]:
%%time
### cell 16 ###

course

In [ ]:
%%time
### cell 17 ###

course.info()

In [ ]:
%%time
### cell 18 ###

course_gp = course.groupby("course_unit").mean(numeric_only=True)

In [ ]:
%%time
### cell 19 ###

course_gp.head()

In [ ]:
%%time
### cell 20 ###

course["course_rating_mean"] = None
for i in course_gp.index:
    course.loc[i, "course_rating_mean"] = course_gp.loc[i, "course_rating_int"]

In [ ]:
%%time
### cell 21 ###

course

In [ ]:
%%time
### cell 22 ###

course.reset_index(inplace=True)

In [ ]:
%%time
### cell 23 ###

course

In [ ]:
%%time
### cell 24 ###

course.groupby("course_unit")["course_rating_int"].mean()

In [ ]:
%%time
### cell 25 ###


def extract_features_from_dataframe(df, group_column, agg_function="mean"):
    """Extracts features for predicting the execution time of a groupby operation."""
    n_rows = df.shape[0]
    n_cols = df.shape[1]

    n_groups = df[group_column].nunique()
    group_sizes = df[group_column].value_counts()
    max_group_size = group_sizes.max()

    int_count = df.select_dtypes(include=["int", "int32", "int64"]).shape[1]
    float_count = df.select_dtypes(include=["float", "float32", "float64"]).shape[1]
    str_count = df.select_dtypes(include=["object", "string"]).shape[1]

    aggregation_mapping = {"sum": 0, "mean": 1, "count": 2}
    agg_function_encoded = aggregation_mapping.get(agg_function, 1)  # Default to 'mean'

    return {
        "n_rows": n_rows,
        "n_cols": n_cols,
        "group_range": n_groups,  # Alias for n_groups
        "n_groups": n_groups,
        "max_group_size": max_group_size,
        "int_count": int_count,
        "float_count": float_count,
        "str_count": str_count,
        "agg_function": agg_function_encoded,
    }


extract_features_from_dataframe(course, "course_unit", "mean")

In [ ]:
%%time
### cell 26 ###

course[course["course_code"].str.startswith("CS")].value_counts()

In [ ]:
%%time
### cell 27 ###

course

In [ ]:
%%time
### cell 28 ###

course = course.sort_values("course_rating_mean", ascending=False)

In [ ]:
%%time
### cell 29 ###

course.reset_index(inplace=True)

In [ ]:
%%time
### cell 30 ###

course.set_index("course_unit", inplace=True)

In [ ]:
%%time
### cell 31 ###

course.loc["KOREA", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 32 ###

KOREA = course[course.index == "KOREA"].squeeze()

In [ ]:
%%time
### cell 33 ###

course.loc["CHINA", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 34 ###

### cell 35 optimized ###
# Use a GPU‐accelerated boolean mask instead of .loc (which falls back to CPU for string labels)
china = course[course.index == "CHINA"].squeeze()

In [ ]:
%%time
### cell 35 ###

course.loc["CHINA", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 36 ###

# Select the "SPAN" row using a GPU‐accelerated boolean mask
span = course[course.index == "SPAN"]

In [ ]:
%%time
### cell 37 ###

course.loc["CS", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 38 ###

cs = course.loc["CS", :]
cs

In [ ]:
%%time
### cell 39 ###

cs_mean = (
    cs.groupby("course_code")
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)
cs_mean

In [ ]:
%%time
### cell 40 ###

course.loc["WKRPT", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 41 ###

# Use GPU-friendly boolean indexing instead of CPU .loc
wkrpt = course[course.index == "WKRPT"]
wkrpt

In [ ]:
%%time
### cell 42 ###

wkrpt_mean = (
    wkrpt.groupby("course_code")
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)
wkrpt_mean

In [ ]:
%%time
### cell 43 ###

wkrpt.groupby("course_code").mean(numeric_only=True)

In [ ]:
%%time
### cell 44 ###

course.loc["PD", "course_rating_mean"].value_counts()

In [ ]:
%%time
### cell 45 ###

# Use GPU-friendly boolean indexing instead of CPU .loc
pd = course[course.index == "PD"]
pd

In [ ]:
%%time
### cell 46 ###

pd_mean = (
    pd.groupby("course_code")
    .mean(numeric_only=True)
    .sort_values("course_rating_int", ascending=False)
)
pd_mean